# 07 — Proper Multi-Fold WC Cross-Validation Tuning

**Goal**: Find hyperparameters that generalize across World Cup years using proper CV.

**CV Strategy**:
- Folds: 2014 WC, 2018 WC, 2022 WC
- For each fold: train on ALL other rows (including post-fold data), test on that WC
- Tune to maximize **average exact score accuracy** across all 3 folds
- Features are pre-match snapshots → no leakage from using post-fold training data

**Outputs**:
- Best LightGBM hyperparameters → update `lgbm_model.py` defaults
- Best XGBoost hyperparameters → update `xgb_model.py` defaults
- Best ensemble weights → use in `train_production.py`

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from src.models.base import load_model_dataset
from src.features.feature_columns import FEATURE_COLS
from src.models.world_cup_utils import create_wc_cv_splits
from src.models.optuna_tuning import (
    LGBMTuner, XGBTuner, PoissonTuner, WCCVTuner, exact_score_accuracy, result_accuracy
)
from src.models.lgbm_model import LGBMGoalModel
from src.models.xgb_model import XGBGoalModel
from src.models.poisson_model import PoissonGoalModel
from src.models.ensemble import EnsembleGoalModel
from src.models.score_conversion import win_draw_loss_probs
from src.evaluation.metrics import rps_batch
from src.models.weighting import apply_competition_weights

TARGET_COLS = ['goals_A', 'goals_B']
CV_YEARS = [2014, 2018, 2022]

print('Imports OK')

Imports OK


## 1. Load Data & Create CV Splits

In [2]:
df = load_model_dataset()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'Dataset: {len(df)} matches, {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Features: {len(FEATURE_COLS)}')

# Validate features present
missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print(f'WARNING: Missing features: {missing}')
else:
    print('All 21 production features present ✓')

# Check WC years available
if 'tournament_year' in df.columns:
    wc_df = df[df['competition'] == 'World Cup']
    print(f'WC matches by year: {wc_df["tournament_year"].value_counts().sort_index().to_dict()}')

Dataset: 21539 matches, 2004-01-01 to 2026-05-16
Features: 20
All 21 production features present ✓
WC matches by year: {2006: 64, 2010: 64, 2014: 64, 2018: 64, 2022: 64}


In [ ]:
df

In [3]:
# Competition weights
weights = apply_competition_weights(df)
print(f'Sample weights: min={weights.min():.3f}, max={weights.max():.3f}, mean={weights.mean():.3f}')

# Create CV splits (val = actual WC matches only, train = everything else)
cv_splits = create_wc_cv_splits(df, CV_YEARS, FEATURE_COLS, TARGET_COLS)

# Build per-fold weight arrays using the SAME mask as create_wc_cv_splits
weights_per_fold = {}
for year in CV_YEARS:
    val_mask = (
        (df['tournament_year'] == year) &
        (df['competition'].str.strip().str.lower() == 'world cup')
    )
    weights_per_fold[year] = weights[~val_mask]
    print(f"  WC {year}: weight array shape = {weights_per_fold[year].shape}")

Sample weights: min=0.685, max=2.739, mean=1.000
WC 2014 fold: train=21475, val=64 (WC only)
WC 2018 fold: train=21475, val=64 (WC only)
WC 2022 fold: train=21475, val=64 (WC only)
  WC 2014: weight array shape = (21475,)
  WC 2018: weight array shape = (21475,)
  WC 2022: weight array shape = (21475,)


## 2. Baseline — Untuned Models

In [4]:
def eval_model_cv(model_class, cv_splits, weights_per_fold=None, **kwargs):
    """Evaluate a model on all CV folds and return per-fold + avg metrics."""
    rows = []
    for year, (X_tr, y_tr, X_val, y_val) in cv_splits.items():
        w = (weights_per_fold or {}).get(year)
        model = model_class(**kwargs)
        model.fit(X_tr, y_tr, sample_weight=w)
        y_pred = np.clip(model.predict(X_val), 0, None)
        exact = exact_score_accuracy(y_val, y_pred)
        result = result_accuracy(y_val, y_pred)
        probs = np.array([win_draw_loss_probs(a, b) for a, b in y_pred])
        rps = rps_batch(y_val, probs)
        rows.append({'year': year, 'exact': exact, 'result': result, 'rps': rps, 'n': len(y_val)})
    
    result_df = pd.DataFrame(rows)
    avg = result_df[['exact', 'result', 'rps']].mean()
    avg_row = pd.DataFrame([{'year': 'avg', 'exact': avg['exact'], 'result': avg['result'], 'rps': avg['rps'], 'n': ''}])
    return pd.concat([result_df, avg_row], ignore_index=True)

print('=== Poisson (default alpha=10) ===')
poisson_baseline = eval_model_cv(PoissonGoalModel, cv_splits, weights_per_fold)
print(poisson_baseline[['year', 'exact', 'result', 'rps']].to_string(index=False))

print('\n=== LightGBM (default params, overfit to 2022) ===')
lgbm_baseline = eval_model_cv(LGBMGoalModel, cv_splits, weights_per_fold)
print(lgbm_baseline[['year', 'exact', 'result', 'rps']].to_string(index=False))

print('\n=== XGBoost (default params) ===')
xgb_baseline = eval_model_cv(XGBGoalModel, cv_splits, weights_per_fold)
print(xgb_baseline[['year', 'exact', 'result', 'rps']].to_string(index=False))

=== Poisson (default alpha=10) ===
year    exact   result      rps
2014 0.234375 0.765625 0.133278
2018 0.218750 0.781250 0.127097
2022 0.171875 0.718750 0.142158
 avg 0.208333 0.755208 0.134178

=== LightGBM (default params, overfit to 2022) ===
year    exact   result      rps
2014 0.203125 0.937500 0.068460
2018 0.265625 0.921875 0.060325
2022 0.265625 0.937500 0.070457
 avg 0.244792 0.932292 0.066414

=== XGBoost (default params) ===
year    exact   result      rps
2014 0.250000 0.937500 0.082007
2018 0.281250 0.906250 0.072508
2022 0.250000 0.937500 0.081311
 avg 0.260417 0.927083 0.078609


## 3. Tune LightGBM with WC CV

In [5]:
N_TRIALS = 150  # Increase for better results; 150 is a good balance
TIMEOUT = 2100  # 30 minutes max

print(f'Tuning LightGBM over {N_TRIALS} trials (CV across {CV_YEARS})...')
lgbm_cv_tuner = WCCVTuner(
    base_tuner_class=LGBMTuner,
    cv_splits=cv_splits,
    weights_per_fold=weights_per_fold,
)
lgbm_result = lgbm_cv_tuner.optimize(n_trials=N_TRIALS, timeout=TIMEOUT, verbose=True)

Tuning LightGBM over 150 trials (CV across [2014, 2018, 2022])...


  0%|          | 0/150 [00:00<?, ?it/s]


Best avg exact score accuracy across folds: 0.2917
Best parameters: {'n_estimators': 276, 'max_depth': 9, 'learning_rate': 0.02597955300094567, 'num_leaves': 233, 'min_child_samples': 33, 'subsample': 0.7334473346895813, 'colsample_bytree': 0.835882051416841, 'reg_alpha': 0.07980827874410094, 'reg_lambda': 0.0017299303935923262}


In [6]:
print('Best LightGBM hyperparameters:')
for k, v in lgbm_result['best_params'].items():
    print(f'  {k}: {v}')
print(f'\nBest avg exact score accuracy: {lgbm_result["best_avg_acc"]*100:.2f}%')

# Evaluate tuned LightGBM on each fold
print('\nPer-fold evaluation (tuned LightGBM):')
tuned_lgbm = LGBMGoalModel(**lgbm_result['best_params'])
for year, (X_tr, y_tr, X_val, y_val) in cv_splits.items():
    w = weights_per_fold.get(year)
    m = LGBMGoalModel(**lgbm_result['best_params'])
    m.fit(X_tr, y_tr, sample_weight=w)
    y_pred = np.clip(m.predict(X_val), 0, None)
    exact = exact_score_accuracy(y_val, y_pred)
    result = result_accuracy(y_val, y_pred)
    probs = np.array([win_draw_loss_probs(a, b) for a, b in y_pred])
    rps = rps_batch(y_val, probs)
    print(f'  WC {year}: exact={exact*100:.1f}%  result={result*100:.1f}%  RPS={rps:.4f}')

Best LightGBM hyperparameters:
  n_estimators: 276
  max_depth: 9
  learning_rate: 0.02597955300094567
  num_leaves: 233
  min_child_samples: 33
  subsample: 0.7334473346895813
  colsample_bytree: 0.835882051416841
  reg_alpha: 0.07980827874410094
  reg_lambda: 0.0017299303935923262

Best avg exact score accuracy: 29.17%

Per-fold evaluation (tuned LightGBM):
  WC 2014: exact=26.6%  result=95.3%  RPS=0.0712
  WC 2018: exact=31.2%  result=90.6%  RPS=0.0642
  WC 2022: exact=29.7%  result=93.8%  RPS=0.0693


## 4. Tune XGBoost with WC CV

In [7]:
print(f'Tuning XGBoost over {N_TRIALS} trials (CV across {CV_YEARS})...')
xgb_cv_tuner = WCCVTuner(
    base_tuner_class=XGBTuner,
    cv_splits=cv_splits,
    weights_per_fold=weights_per_fold,
)
xgb_result = xgb_cv_tuner.optimize(n_trials=N_TRIALS, timeout=TIMEOUT, verbose=True)

Tuning XGBoost over 150 trials (CV across [2014, 2018, 2022])...


  0%|          | 0/150 [00:00<?, ?it/s]


Best avg exact score accuracy across folds: 0.2812
Best parameters: {'n_estimators': 268, 'max_depth': 8, 'learning_rate': 0.055263064405123206, 'subsample': 0.9888455338600703, 'colsample_bytree': 0.9528625653858853, 'gamma': 2.3702565701276304, 'min_child_weight': 9, 'reg_alpha': 0.1400481305977997, 'reg_lambda': 2.2312748382635074e-05}


In [8]:
print('Best XGBoost hyperparameters:')
for k, v in xgb_result['best_params'].items():
    print(f'  {k}: {v}')
print(f'\nBest avg exact score accuracy: {xgb_result["best_avg_acc"]*100:.2f}%')

print('\nPer-fold evaluation (tuned XGBoost):')
for year, (X_tr, y_tr, X_val, y_val) in cv_splits.items():
    w = weights_per_fold.get(year)
    m = XGBGoalModel(**xgb_result['best_params'])
    m.fit(X_tr, y_tr, sample_weight=w)
    y_pred = np.clip(m.predict(X_val), 0, None)
    exact = exact_score_accuracy(y_val, y_pred)
    result = result_accuracy(y_val, y_pred)
    probs = np.array([win_draw_loss_probs(a, b) for a, b in y_pred])
    rps = rps_batch(y_val, probs)
    print(f'  WC {year}: exact={exact*100:.1f}%  result={result*100:.1f}%  RPS={rps:.4f}')

Best XGBoost hyperparameters:
  n_estimators: 268
  max_depth: 8
  learning_rate: 0.055263064405123206
  subsample: 0.9888455338600703
  colsample_bytree: 0.9528625653858853
  gamma: 2.3702565701276304
  min_child_weight: 9
  reg_alpha: 0.1400481305977997
  reg_lambda: 2.2312748382635074e-05

Best avg exact score accuracy: 28.12%

Per-fold evaluation (tuned XGBoost):
  WC 2014: exact=29.7%  result=95.3%  RPS=0.0700
  WC 2018: exact=26.6%  result=90.6%  RPS=0.0607
  WC 2022: exact=28.1%  result=90.6%  RPS=0.0708


## 5. Ensemble Weight Search

In [9]:
# Grid search over LGB/XGB/Poisson ensemble weights
weight_grid = []
for w_lgb in np.arange(0.0, 1.05, 0.1):
    for w_xgb in np.arange(0.0, 1.05 - w_lgb, 0.1):
        w_pois = round(1.0 - w_lgb - w_xgb, 2)
        if w_pois < -0.01:
            continue
        weight_grid.append((round(w_lgb, 2), round(w_xgb, 2), max(w_pois, 0.0)))

print(f'Testing {len(weight_grid)} weight combinations...')

best_ensemble = {'avg_exact': 0.0, 'w_lgb': 0, 'w_xgb': 0, 'w_pois': 0}
ensemble_results = []

for w_lgb, w_xgb, w_pois in weight_grid:
    if w_lgb + w_xgb + w_pois < 0.99:
        continue
    fold_exacts = []
    for year, (X_tr, y_tr, X_val, y_val) in cv_splits.items():
        wt = weights_per_fold.get(year)
        models, wts = [], []
        if w_lgb > 0:
            m = LGBMGoalModel(**lgbm_result['best_params'])
            m.fit(X_tr, y_tr, sample_weight=wt)
            models.append(m); wts.append(w_lgb)
        if w_xgb > 0:
            m = XGBGoalModel(**xgb_result['best_params'])
            m.fit(X_tr, y_tr, sample_weight=wt)
            models.append(m); wts.append(w_xgb)
        if w_pois > 0:
            m = PoissonGoalModel()
            m.fit(X_tr, y_tr, sample_weight=wt)
            models.append(m); wts.append(w_pois)
        ensemble = EnsembleGoalModel(models, weights=wts)
        y_pred = np.clip(ensemble.predict(X_val), 0, None)
        fold_exacts.append(exact_score_accuracy(y_val, y_pred))
    avg = np.mean(fold_exacts)
    ensemble_results.append({'w_lgb': w_lgb, 'w_xgb': w_xgb, 'w_pois': w_pois, 'avg_exact': avg})
    if avg > best_ensemble['avg_exact']:
        best_ensemble = {'avg_exact': avg, 'w_lgb': w_lgb, 'w_xgb': w_xgb, 'w_pois': w_pois}

ens_df = pd.DataFrame(ensemble_results).sort_values('avg_exact', ascending=False)
print('Top 10 ensemble configurations:')
print(ens_df.head(10).to_string(index=False))
print(f'\nChampion: LGB={best_ensemble["w_lgb"]} XGB={best_ensemble["w_xgb"]} Pois={best_ensemble["w_pois"]}')
print(f'Avg exact score: {best_ensemble["avg_exact"]*100:.2f}%')

Testing 66 weight combinations...
Top 10 ensemble configurations:
 w_lgb  w_xgb  w_pois  avg_exact
   0.9    0.1    -0.0   0.302083
   1.0    0.0     0.0   0.291667
   0.8    0.2    -0.0   0.291667
   0.7    0.2     0.1   0.291667
   0.6    0.3     0.1   0.291667
   0.8    0.1     0.1   0.286458
   0.3    0.6     0.1   0.286458
   0.2    0.7     0.1   0.286458
   0.4    0.5     0.1   0.286458
   0.7    0.3    -0.0   0.286458

Champion: LGB=0.9 XGB=0.1 Pois=-0.0
Avg exact score: 30.21%


## 6. Final Summary — Best Parameters to Use in Production

In [10]:
print('='*70)
print('FINAL RESULTS SUMMARY')
print('='*70)
print(f'\nLightGBM best params (copy to lgbm_model.py defaults):')
print(f'  {lgbm_result["best_params"]}')
print(f'  → Avg exact score CV: {lgbm_result["best_avg_acc"]*100:.2f}%')

print(f'\nXGBoost best params (copy to xgb_model.py defaults):')
print(f'  {xgb_result["best_params"]}')
print(f'  → Avg exact score CV: {xgb_result["best_avg_acc"]*100:.2f}%')

print(f'\nBest ensemble weights:')
print(f'  LGB={best_ensemble["w_lgb"]}, XGB={best_ensemble["w_xgb"]}, Poisson={best_ensemble["w_pois"]}')
print(f'  → Avg exact score CV: {best_ensemble["avg_exact"]*100:.2f}%')

print(f'\nBaseline comparisons (avg across 2014/2018/2022 WC folds):')
poisson_avg = float(poisson_baseline[poisson_baseline['year']=='avg']['exact'].iloc[0])
lgbm_base_avg = float(lgbm_baseline[lgbm_baseline['year']=='avg']['exact'].iloc[0])
xgb_base_avg = float(xgb_baseline[xgb_baseline['year']=='avg']['exact'].iloc[0])
print(f'  Poisson (default):   {poisson_avg*100:.2f}%')
print(f'  LightGBM (default):  {lgbm_base_avg*100:.2f}%')
print(f'  XGBoost (default):   {xgb_base_avg*100:.2f}%')
print(f'  LightGBM (tuned CV): {lgbm_result["best_avg_acc"]*100:.2f}%')
print(f'  XGBoost (tuned CV):  {xgb_result["best_avg_acc"]*100:.2f}%')
print(f'  Ensemble (best):     {best_ensemble["avg_exact"]*100:.2f}%')

FINAL RESULTS SUMMARY

LightGBM best params (copy to lgbm_model.py defaults):
  {'n_estimators': 276, 'max_depth': 9, 'learning_rate': 0.02597955300094567, 'num_leaves': 233, 'min_child_samples': 33, 'subsample': 0.7334473346895813, 'colsample_bytree': 0.835882051416841, 'reg_alpha': 0.07980827874410094, 'reg_lambda': 0.0017299303935923262}
  → Avg exact score CV: 29.17%

XGBoost best params (copy to xgb_model.py defaults):
  {'n_estimators': 268, 'max_depth': 8, 'learning_rate': 0.055263064405123206, 'subsample': 0.9888455338600703, 'colsample_bytree': 0.9528625653858853, 'gamma': 2.3702565701276304, 'min_child_weight': 9, 'reg_alpha': 0.1400481305977997, 'reg_lambda': 2.2312748382635074e-05}
  → Avg exact score CV: 28.12%

Best ensemble weights:
  LGB=0.9, XGB=0.1, Poisson=-0.0
  → Avg exact score CV: 30.21%

Baseline comparisons (avg across 2014/2018/2022 WC folds):
  Poisson (default):   20.83%
  LightGBM (default):  24.48%
  XGBoost (default):   26.04%
  LightGBM (tuned CV): 29.17

## 7. Update Model Defaults

After reviewing the results above, manually copy the best params to:
- `src/models/lgbm_model.py` — update `__init__` defaults
- `src/models/xgb_model.py` — update `__init__` defaults

Then run the production training script:
```bash
python -m src.models.train_production --model-type lgbm
```
or for ensemble:
```bash
python -m src.models.train_production --model-type ensemble
```

## 8. Ensemble Backtest — 2022 WC Match-by-Match

Train ensemble on all non-2022-WC data, then predict each 2022 WC match in order.
Uses pre-computed state features from the dataset (which were computed chronologically — no leakage).

In [11]:
from src.models.score_conversion import top_scores as get_top_scores

# --- Build 2022 WC fold data ---
X_train_22, y_train_22, X_val_22, y_val_22 = cv_splits[2022]
w_train_22 = weights_per_fold[2022]

# Metadata for the 2022 WC matches (for display)
wc22_mask = (
    (df['tournament_year'] == 2022) &
    (df['competition'].str.strip().str.lower() == 'world cup')
)
wc22_meta = df[wc22_mask][['date', 'team_A', 'team_B']].reset_index(drop=True)

# --- Train ensemble with best weights from tuning ---
# UPDATE these weights to whatever best_ensemble showed in cell 5
W_LGB  = best_ensemble['w_lgb']
W_XGB  = best_ensemble['w_xgb']
W_POIS = best_ensemble['w_pois']

print(f"Ensemble weights: LGB={W_LGB}, XGB={W_XGB}, Poisson={W_POIS}")
print(f"Training on {len(X_train_22)} matches...")

models_ens, wts_ens = [], []
if W_LGB > 0:
    m = LGBMGoalModel(**lgbm_result['best_params'])
    m.fit(X_train_22, y_train_22, sample_weight=w_train_22)
    models_ens.append(m); wts_ens.append(W_LGB)
if W_XGB > 0:
    m = XGBGoalModel(**xgb_result['best_params'])
    m.fit(X_train_22, y_train_22, sample_weight=w_train_22)
    models_ens.append(m); wts_ens.append(W_XGB)
if W_POIS > 0:
    m = PoissonGoalModel()
    m.fit(X_train_22, y_train_22, sample_weight=w_train_22)
    models_ens.append(m); wts_ens.append(W_POIS)

ensemble_model = EnsembleGoalModel(models_ens, weights=wts_ens)
print("Ensemble trained.")

# --- Predict all 64 WC 2022 matches ---
y_pred_22 = np.clip(ensemble_model.predict(X_val_22), 0, None)
y_pred_rounded = np.round(y_pred_22).astype(int)

# --- Build per-match results table ---
rows = []
for i in range(len(y_val_22)):
    actual_a, actual_b = int(y_val_22[i, 0]), int(y_val_22[i, 1])
    pred_a, pred_b     = int(y_pred_rounded[i, 0]), int(y_pred_rounded[i, 1])
    lambda_a, lambda_b = float(y_pred_22[i, 0]), float(y_pred_22[i, 1])

    actual_result = 'W' if actual_a > actual_b else ('D' if actual_a == actual_b else 'L')
    pred_result   = 'W' if pred_a   > pred_b   else ('D' if pred_a   == pred_b   else 'L')
    exact_ok      = (actual_a == pred_a) and (actual_b == pred_b)
    result_ok     = actual_result == pred_result

    best_score = get_top_scores(lambda_a, lambda_b, n=1)[0]  # (ga, gb, prob)

    rows.append({
        'date':      wc22_meta.iloc[i]['date'],
        'team_A':    wc22_meta.iloc[i]['team_A'],
        'team_B':    wc22_meta.iloc[i]['team_B'],
        'actual':    f"{actual_a}-{actual_b}",
        'predicted': f"{pred_a}-{pred_b}",
        'lambda':    f"{lambda_a:.2f}-{lambda_b:.2f}",
        'top_score': f"{best_score[0]}-{best_score[1]} ({best_score[2]*100:.1f}%)",
        'exact':     exact_ok,
        'result':    result_ok,
    })

results_22 = pd.DataFrame(rows)

n_exact  = results_22['exact'].sum()
n_result = results_22['result'].sum()
n_total  = len(results_22)
print(f"\n2022 WC Ensemble Backtest ({n_total} matches)")
print(f"  Exact score:    {n_exact}/{n_total}  ({n_exact/n_total*100:.1f}%)")
print(f"  Result W/D/L:   {n_result}/{n_total}  ({n_result/n_total*100:.1f}%)")
print(f"  Goal MAE (exp): {np.mean(np.abs(y_val_22 - y_pred_22)):.3f}")

Ensemble weights: LGB=0.9, XGB=0.1, Poisson=-0.0
Training on 21475 matches...
Ensemble trained.

2022 WC Ensemble Backtest (64 matches)
  Exact score:    19/64  (29.7%)
  Result W/D/L:   60/64  (93.8%)
  Goal MAE (exp): 0.724


In [13]:
%pip install jinja2

Note: you may need to restart the kernel to use updated packages.


In [13]:
# Full match-by-match table
%pip install jinja2
pd.set_option('display.max_rows', 70)
pd.set_option('display.width', 130)
display(results_22.sort_values('date').style
    .apply(lambda col: ['background-color: #d4edda' if v else '' for v in col]
           if col.name == 'exact' else
           ['background-color: #d4edda' if v else 'background-color: #f8d7da' for v in col]
           if col.name == 'result' else
           ['' for _ in col], axis=0)
)

# Summary breakdown
print("\nResult breakdown:")
print(f"  Wrong result (missed): {(~results_22['result']).sum()} matches")
print(f"  Right result, wrong score: {(results_22['result'] & ~results_22['exact']).sum()} matches")
print(f"  Right exact score: {results_22['exact'].sum()} matches")

print("\nDraw prediction accuracy:")
actual_draws = results_22['actual'].apply(lambda s: s.split('-')[0] == s.split('-')[1])
pred_draws   = results_22['predicted'].apply(lambda s: s.split('-')[0] == s.split('-')[1])
print(f"  Actual draws: {actual_draws.sum()}")
print(f"  Predicted draws: {pred_draws.sum()}")
print(f"  Correctly predicted draws: {(actual_draws & pred_draws).sum()}")

Note: you may need to restart the kernel to use updated packages.


AttributeError: The '.style' accessor requires jinja2